# 08 – NLP Analysis

### Purpose of the Notebook
Analyse textbasierter Vergabefelder mittels NLP.

### Steps
- TF‑IDF preprocessing
- SVD dimensionality reduction
- NMF topic modelling
- SVM text‑risk classifier (3 classes)
- Export TEXT_RISK_SCORE + NLP features
- Integration into modelling pipeline

--------------------
#### Imports & Setup & Dataset
-------------------

In [22]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

import pickle

In [2]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [6]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.nlp_processing import (text_preproceccing, build_tfidf_svd, build_nmf_topics,
                               build_text_risk_classifier, predict_text_risk)

from my_scripts.eda import (overview, filter_germany)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset_comp.pkl")

print("EU dataset:", df.shape)

EU dataset: (4039906, 39)


----------------
### NLP PROCESSING

----------

In [8]:
# ---------------------------------------------------------
# Preprocess text 
# ---------------------------------------------------------

df = text_preproceccing(df, ["TEXT_ALL"])


In [9]:
df.shape

(4039906, 39)

In [10]:
# ---------------------------------------------------------
# TF‑IDF + SVD features
# ---------------------------------------------------------

df_svd, X_tfidf, tfidf_vectorizer, svd_model = build_tfidf_svd(df["TEXT_ALL"])
df = pd.concat([df, df_svd], axis=1)


In [11]:
# ---------------------------------------------------------
# Topic modelling (NMF)
# ---------------------------------------------------------
df_topics, nmf_model = build_nmf_topics(X_tfidf)
df = pd.concat([df, df_topics], axis=1)


In [12]:
df.shape

(4039906, 154)

In [13]:
# ---------------------------------------------------------
# Prepare labels for TEXT_RISK_SCORE
# ---------------------------------------------------------

risk_map = {
    "failed": "high",
    "low": "medium",
    "medium": "medium",
    "high": "low"
}

df["TEXT_RISK_LABEL"] = df["OFFERS_BIN"].map(risk_map)


In [14]:
# ---------------------------------------------------------
# Train SVM classifier
# ---------------------------------------------------------

df_text = df[df["TEXT_RISK_LABEL"].notna()].copy()

texts = df_text["TEXT_ALL"].fillna("").astype(str)
labels = df_text["TEXT_RISK_LABEL"].astype(str)

svm_model, tfidf_svm, label_encoder = build_text_risk_classifier(texts, labels)



In [15]:
# ---------------------------------------------------------
# Predict TEXT_RISK_SCORE
# ---------------------------------------------------------

df["TEXT_RISK_SCORE"] = predict_text_risk(
    df["TEXT_ALL"],
    svm_model,
    tfidf_svm,
    label_encoder
)



In [16]:
df.head()

,YEAR,ID_TYPE,XSD_VERSION,CANCELLED,CORRECTIONS,ISO_COUNTRY_CODE,CAE_TYPE,B_AWARDED_BY_CENTRAL_BODY,TYPE_OF_CONTRACT,TAL_LOCATION_NUTS,B_DYN_PURCH_SYST,ID_LOT,B_EU_FUNDS,TOP_TYPE,B_ACCELERATED,OUT_OF_DIRECTIVES,CRIT_CODE,CRIT_PRICE_WEIGHT,B_ELECTRONIC_AUCTION,NUMBER_AWARDS,B_AWARDED_TO_A_GROUP,WIN_COUNTRY_CODE,B_CONTRACTOR_SME,B_SUBCONTRACTED,TEXT_ALL,AWARD_QUARTER,DAYS_TO_AWARD,CPV_DIVISION,CPV_GROUP,CPV_CLASS,IS_FAILED_TENDER,IS_LOW_COMPETITION,OFFERS_BIN,VALUE_BIN,HAS_MULTIPLE_LOTS,LOTS_BIN,VALUE_EURO_MISSING,AWARD_VALUE_EURO_MISSING,NUMBER_OFFERS_MISSING,NLP_SVD_0,NLP_SVD_1,NLP_SVD_2,NLP_SVD_3,NLP_SVD_4,NLP_SVD_5,NLP_SVD_6,NLP_SVD_7,NLP_SVD_8,NLP_SVD_9,NLP_SVD_10,NLP_SVD_11,NLP_SVD_12,NLP_SVD_13,NLP_SVD_14,NLP_SVD_15,NLP_SVD_16,NLP_SVD_17,NLP_SVD_18,NLP_SVD_19,NLP_SVD_20,NLP_SVD_21,NLP_SVD_22,NLP_SVD_23,NLP_SVD_24,NLP_SVD_25,NLP_SVD_26,NLP_SVD_27,NLP_SVD_28,NLP_SVD_29,NLP_SVD_30,NLP_SVD_31,NLP_SVD_32,NLP_SVD_33,NLP_SVD_34,NLP_SVD_35,NLP_SVD_36,NLP_SVD_37,NLP_SVD_38,NLP_SVD_39,NLP_SVD_40,NLP_SVD_41,NLP_SVD_42,NLP_SVD_43,NLP_SVD_44,NLP_SVD_45,NLP_SVD_46,NLP_SVD_47,NLP_SVD_48,NLP_SVD_49,NLP_SVD_50,NLP_SVD_51,NLP_SVD_52,NLP_SVD_53,NLP_SVD_54,NLP_SVD_55,NLP_SVD_56,NLP_SVD_57,NLP_SVD_58,NLP_SVD_59,NLP_SVD_60,NLP_SVD_61,NLP_SVD_62,NLP_SVD_63,NLP_SVD_64,NLP_SVD_65,NLP_SVD_66,NLP_SVD_67,NLP_SVD_68,NLP_SVD_69,NLP_SVD_70,NLP_SVD_71,NLP_SVD_72,NLP_SVD_73,NLP_SVD_74,NLP_SVD_75,NLP_SVD_76,NLP_SVD_77,NLP_SVD_78,NLP_SVD_79,NLP_SVD_80,NLP_SVD_81,NLP_SVD_82,NLP_SVD_83,NLP_SVD_84,NLP_SVD_85,NLP_SVD_86,NLP_SVD_87,NLP_SVD_88,NLP_SVD_89,NLP_SVD_90,NLP_SVD_91,NLP_SVD_92,NLP_SVD_93,NLP_SVD_94,NLP_SVD_95,NLP_SVD_96,NLP_SVD_97,NLP_SVD_98,NLP_SVD_99,NLP_TOPIC_0,NLP_TOPIC_1,NLP_TOPIC_2,NLP_TOPIC_3,NLP_TOPIC_4,NLP_TOPIC_5,NLP_TOPIC_6,NLP_TOPIC_7,NLP_TOPIC_8,NLP_TOPIC_9,NLP_TOPIC_10,NLP_TOPIC_11,NLP_TOPIC_12,NLP_TOPIC_13,NLP_TOPIC_14,TEXT_RISK_LABEL,TEXT_RISK_SCORE
0,2008,3,D205,0,0,DE,8,Unknown,W,DED31,0,Unknown,Unknown,OPE,0,0,M,NaN,0,1,0,DE,0,Unknown,unknown preis qualit t 60 40,3.00,-73.00,45,452,4521,0,1,low,NaN,0,single,1,0,0,0.07,0.20,0.02,-0.25,0.09,-0.04,0.37,-0.32,-0.04,-0.00,-0.03,0.14,0.16,0.01,-0.02,0.03,-0.04,-0.09,0.09,-0.02,-0.02,0.02,0.14,-0.07,-0.18,-0.03,0.10,-0.11,0.06,-0.09,0.07,-0.03,-0.03,-0.07,0.02,0.00,-0.01,0.04,-0.01,-0.01,-0.01,-0.03,0.03,0.03,0.02,-0.06,-0.03,-0.05,-0.03,0.01,-0.01,0.04,-0.01,0.05,0.01,-0.03,0.04,0.05,-0.05,-0.02,-0.02,-0.03,0.02,0.05,0.02,-0.01,0.02,0.01,0.01,-0.04,-0.07,0.11,0.01,0.05,0.06,-0.11,0.07,0.08,-0.03,0.05,0.03,-0.03,-0.04,0.08,-0.04,0.04,-0.09,-0.11,0.08,0.00,-0.00,-0.03,0.02,-0.00,-0.01,0.02,0.01,0.06,0.02,-0.04,0.00,0.00,0.00,0.00,0.00,0.00,0.04,0.00,0.00,0.00,0.00,0.03,0.00,0.00,0.00,medium,low
1,2008,3,D205,0,0,DE,3,Unknown,W,DE913,0,Unknown,N,OPE,0,0,L,100.00,0,1,0,DE,0,N,unknown unknown unknown,4.00,-6.00,45,452,4521,0,0,medium,NaN,0,single,1,0,0,1.00,-0.01,-0.01,0.00,-0.00,0.00,-0.00,0.00,0.00,-0.00,-0.00,-0.00,0.00,-0.00,-0.00,0.00,-0.00,-0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,0.00,-0.00,0.00,-0.00,0.00,0.00,-0.00,-0.00,-0.00,0.00,-0.00,0.00,-0.00,-0.00,0.00,-0.01,-0.00,-0.00,0.01,0.00,0.00,0.00,-0.00,0.00,0.00,-0.00,0.00,0.00,0.00,-0.00,0.00,0.00,0.00,0.00,0.00,-0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,-0.00,0.00,0.00,-0.00,-0.00,0.00,0.00,0.00,-0.00,0.00,0.00,-0.00,0.00,0.00,-0.00,0.00,-0.00,0.00,0.00,0.00,-0.00,0.03,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,medium,medium
2,2008,3,D205,0,0,FR,3,Unknown,W,Unknown,0,Unknown,N,OPE,0,0,M,NaN,0,1,0,FR,0,N,unknown prix d lai d intervention d urgence 80 20,4.00,-37.00,45,454,4542,1,0,failed,NaN,0,single,1,1,0,0.06,0.12,0.02,0.00,0.05,0.02,0.03,0.07,-0.00,-0.02,0.16,-0.08,-0.00,0.34,0.04,0.01,-0.04,-0.06,0.02,-0.09,-0.02,-0.03,-0.04,0.07,0.05,-0.05,-0.00,-0.03,0.09,0.02,-0.02,-0.01,-0.01,-0.01,-0.03,0.02,-0.01,-0.01,-0.04,0.08,0.01,-0.09,0.04,0.02,-0.09,-0.12,-0.03,0.04,0.07,0.02,0.03,0.00,0.07,0.07,0.05,-0.07,-0.00,-0.04,0.01,0.01,0.05,0.06,

In [17]:
df.columns

Index(['YEAR', 'ID_TYPE', 'XSD_VERSION', 'CANCELLED', 'CORRECTIONS',
       'ISO_COUNTRY_CODE', 'CAE_TYPE', 'B_AWARDED_BY_CENTRAL_BODY',
       'TYPE_OF_CONTRACT', 'TAL_LOCATION_NUTS',
       ...
       'NLP_TOPIC_7', 'NLP_TOPIC_8', 'NLP_TOPIC_9', 'NLP_TOPIC_10',
       'NLP_TOPIC_11', 'NLP_TOPIC_12', 'NLP_TOPIC_13', 'NLP_TOPIC_14',
       'TEXT_RISK_LABEL', 'TEXT_RISK_SCORE'],
      dtype='str', length=156)

In [20]:
# drop unnecessary column
df = df.drop(columns="TEXT_ALL", errors="ignore").reset_index(drop=True)


#### Notes: NLP Pipeline for Tender Risk Prediction

1. Dataset Size After Features Engineering
- Full EU dataset:
  - Before: 4,039,906 rows × 41 columns
  - After: 4,039,906 rows × 154 columns

2. Text fields provide additional signals not captured by structured variables.  
Short titles and evaluation‑related text (TITLE, CRIT_CRITERIA, CRIT_WEIGHTS AS TEXT_ALL) reveal complexity, niche requirements, and multi‑criteria scoring patterns that strongly influence bidder participation and failure risk.

3. TF‑IDF offers a scalable and domain‑appropriate representation of tender text.  
It efficiently captures important terms and patterns without requiring heavy linguistic models, making it suitable for millions of records and classical ML workflows.

4. Dimensionality reduction (SVD) converts high‑dimensional TF‑IDF vectors into compact numerical features.  
This reduces sparsity, stabilizes downstream models, and enables seamless integration with structured predictors such as CPV, procedure type, and tender value.

5. Topic modelling (NMF) introduces interpretable thematic structure.  
Extracted topics highlight procurement areas with systematically higher failure rates, improving interpretability and analytical insight.

6. A text‑based classifier (TF‑IDF + SVM) produces a high‑level TEXT_RISK_SCORE.  
It learns patterns associated with failed, medium‑risk, and safe tenders based solely on text, generating a categorical risk signal usable even when raw text is unavailable.

7. TEXT_RISK_SCORE is essential for downstream applications such as the risk simulator.  
It allows the model to incorporate text‑derived risk information without requiring free‑text input, enabling scenario simulations based on a small set of structured parameters.

8. The combined pipeline remains interpretable, scalable, and robust.  
TF‑IDF + SVD ensures numerical stability, NMF adds thematic insight, and SVM provides a practical risk score — together forming a balanced NLP module that strengthens the overall tender risk prediction model.

--------------
## Top topics

-------------

In [ ]:
# ---------------------------------------------------------
# General Topics
# ---------------------------------------------------------

#choice topics

feature_names = tfidf_vectorizer.get_feature_names_out()

topics = {}

for i, topic in enumerate(nmf_model.components_):
    top_indices = topic.argsort()[-20:]  # топ 20 слів
    top_words = [feature_names[j] for j in top_indices]
    topics[f"Topic_{i}"] = top_words


In [25]:
# print topics
for t, words in topics.items():
    print(f"{t}: {', '.join(words)}")


Topic_0: pozycja, mg, st, ml, di, lot, grupa unknown, poz, za, na, cz, in, pakiet unknown, und, zadanie, do, dostawa, grupa, unknown unknown, unknown
Topic_1: del, mati re, en mati, mati, prix de, re de, fourniture de, offre prix, lai, lai de, fourniture, de livraison, livraison, technique de, en, de la, de offre, la, offre, de
Topic_2: nr 13, 12, do, nr poz, nr 12, perceel, perceel nr, 11, nr 11, nr 10, nr cena, poz, pakiet unknown, zadanie nr, zadanie, unknown, nr unknown, pakiet nr, nr, pakiet
Topic_3: offre prix, lot, prix 70, 55, technique 40, 45, technique de, de offre, offre, technique 60, unknown, unknown prix, prix valeur, prix 60, unknown valeur, prix, technique prix, valeur technique, technique, valeur
Topic_4: lot, technical, the, cost, 40 60, delivery, price 70, quality 70, unknown, quality 40, of, quality 60, price 60, and, unknown quality, quality price, unknown price, price quality, price, quality
Topic_5: termin realizacji, realizacji, do, 99, ci 95, dostawy 95, 98, 90

#### Notes: Interpreted Topic Categories
1. Topic 0 — Tender Lots & Procurement Packages (PL/NL)
  - Frequent terms: pozycja, grupa, pakiet, zadanie, dostawa, poz, perceel  
  - Meaning: structural elements of tenders (lots, packages, tasks).
  - Category: Tender structure / lot definitions

2. Topic 1 — Technical Supply & Delivery Conditions (FR)
  - Frequent terms: fourniture, livraison, prix de, offre prix, technique de  
  - Meaning: French technical descriptions and delivery specifications.
  - Category: Technical supply & delivery requirements

3. Topic 2 — Lot Numbering & Sub‑lot Structure (PL/NL)
  - Frequent terms: nr, perceel nr, zadanie nr, pakiet nr  
  - Meaning: numbering of lots, sub‑lots, and tender sections.
  - Category: Lot numbering / sub‑lot segmentation

4. Topic 3 — Price vs Technical Value (FR)
  - Frequent terms: offre prix, technique, valeur technique, prix 60  
  - Meaning: French scoring formulas combining price and technical value.
  - Category: Price–technical value scoring

5. Topic 4 — Quality–Price Evaluation (EN)
  - Frequent terms: technical, quality, delivery, price 70, quality 40  
  - Meaning: English‑language tenders evaluating quality and price.
  - Category: Quality–price evaluation (EN tenders)

6. Topic 5 — Delivery Deadlines & Execution Timing (PL)
  - Frequent terms: termin realizacji, dostawy, cena termin, 90/10  
  - Meaning: deadlines, delivery schedules, execution timing.
  - Category: Delivery timelines / execution deadlines

7. Topic 6 — German Technical Criteria (Preis/Qualität)
  - Frequent terms: preis, technischer wert, qualit, los, preis qualit  
  - Meaning: German scoring based on technical value and price.
  - Category: German technical criteria (Preis/Qualität)

8. Topic 7 — Scoring Formulas (Mixed Languages)
  - Frequent terms: 30/10, 40/30, cena, jakość, price 70  
  - Meaning: mixed scoring formulas (30/70, 40/60, 70/30).
  - Category: Scoring ratios (price/quality)

9. Topic 8 — Price–Quality Scoring (IT/FR)
  - Frequent terms: prezzo, qualit, prix 50, 50/50  
  - Meaning: Italian/French scoring formulas.
  - Category: Price–quality scoring (IT/FR)

10. Topic 9 — Pharmaceuticals & Medical Drugs (PL)
  - Frequent terms: leki onkologiczne, psychotropowe, pakiet leki  
  - Meaning: medical and pharmaceutical procurement.
  - Category: Pharmaceutical products / medical drugs

11. Topic 10 — Service & Management Contracts (EN)
  - Frequent terms: delivery, management, service, 40/10, 30/10  
  - Meaning: service‑based tenders, management contracts.
  - Category: Service & management procurement

12. Topic 11 — Quality–Price Evaluation (FR/IT)
  - Frequent terms: prix, qualit, prezzo, technique 60, 40/60  
  - Meaning: quality/price scoring in FR/IT tenders.
  - Category: Quality–price evaluation (FR/IT)

13. Topic 12 — Technical Services & Performance Criteria (FR)
  - Frequent terms: prestations, technique des, valeur technique  
  - Meaning: technical services, performance‑based evaluation.
  - Category: Technical services / performance criteria

14. Topic 13 — High‑Weight Quality Scoring (80/20 etc.)
  - Frequent terms: cena jakość, quality 80, technique 80, 40/40  
  - Meaning: tenders with strong emphasis on quality.
  - Category: High‑weight quality scoring

15. Topic 14 — General Evaluation Criteria (FR)
  - Frequent terms: critère, prix, qualité, lots, de la, pour les  
  - Meaning: general French evaluation criteria.
  - Category: General evaluation criteria (FR)


In [ ]:
# ---------------------------------------------------------
# Top Topics for Germany
# ---------------------------------------------------------

# topics range for Germany

topic_cols = [c for c in df_de.columns if c.startswith("NLP_TOPIC_")]
df_de["dominant_topic"] = df_de[topic_cols].idxmax(axis=1)
df_de["dominant_topic"].value_counts()



dominant_topic
NLP_TOPIC_0     165249
NLP_TOPIC_6      74351
NLP_TOPIC_10     21011
NLP_TOPIC_13     13555
NLP_TOPIC_8      10745
NLP_TOPIC_7       9060
NLP_TOPIC_11      5599
NLP_TOPIC_12      1513
NLP_TOPIC_14      1107
NLP_TOPIC_2        653
NLP_TOPIC_4        265
NLP_TOPIC_5        157
NLP_TOPIC_3         42
NLP_TOPIC_1         40
NLP_TOPIC_9          2
Name: count, dtype: int64

### Germany‑Specific Topic Profile (based on PCA, KMeans, and dominant topic distribution)
1. Tender Lots & Procurement Packages (Topic 0)
2. Technical Criteria (Topic 6)
3. Service & Management Contracts (Topic 10)
4. High‑Weight Quality Scoring (Topic 13)
5. Price–Quality Scoring (IT/FR patterns) (Topic 8)
6. Scoring Formulas (price/quality ratios) (Topic 7)
7. Quality–Price Evaluation (FR/IT) (Topic 11)
8. Technical Services & Performance Criteria (Topic 12)
9. General Evaluation Criteria (Topic 14)
10. Lot Numbering & Sub‑lot Structure (Topic 2)


---------
### SAVE DATASET & MODEL

--------

In [18]:
# Topics for Germany 

df_de = filter_germany(df)

topic_cols = [c for c in df_de.columns if c.startswith("NLP_TOPIC_")]

df_de_topics = df_de[topic_cols].copy()

df_de_topics.to_pickle("../data/dataset_topics_de.pkl")


In [19]:
overview(df_de_topics)

,dtype,total,missing_n,missing_%,uniques_n,uniques
NLP_TOPIC_0,float64,303349,0,0.00,44509,"[0.0003314750934450534, 0.02949100655580227, 0..."
NLP_TOPIC_1,float64,303349,0,0.00,19515,"[0.0, 0.000117022053536669, 0.0001024567302792..."
NLP_TOPIC_2,float64,303349,0,0.00,36768,"[0.0, 0.0001858849475990808, 8.156867794011572..."
NLP_TOPIC_3,float64,303349,0,0.00,18160,"[0.0, 0.00021163279261874498, 0.00181716858834..."
NLP_TOPIC_4,float64,303349,0,0.00,19555,"[0.0, 8.33115794251757e-05, 3.487545585812261e..."
NLP_TOPIC_5,float64,303349,0,0.00,9747,"[0.0, 0.003647037369706218, 0.0113458998938106..."
NLP_TOPIC_6,float64,303349,0,0.00,71203,"[0.03735687554948603, 0.0, 0.02617342574974445..."
NLP_TOPIC_7,float64,303349,0,0.00,40997,"[0.0006765567940184736, 0.0, 0.000238186898063..."
NLP_TOPIC_8,float64,303349,0,0.00,50474,"[0.0019130184716387873, 0.0, 0.000104612869572..."
NLP_TOPIC_9,float64,303349,0,0.00,42903,"[3.2147495535441425e-05, 0.0, 0.00048351957274..."


In [62]:
df.to_pickle("../data/dataset_nlp.pkl")

In [23]:
with open("../models/nmf_model.pkl", "wb") as f:
    pickle.dump(nmf_model, f)

with open("../models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)
